# Analysis of TikTok Digital Nomad Content: Jaccard Co-occurrence of Qualitative Codes


## InterCoder Reliability Ratio

In [ ]:
import pandas as pd

#Open the sample with human codes
combined_human_model_codes = pd.read_csv('../csv/combined_human_model_codes.csv')

In [ ]:
def calculate_jaccard_analysis(df):
    """
    Calculate Jaccard similarity between human and model annotations.
    
    Args:
        df: DataFrame containing human and model annotations
        
    Returns:
        tuple: (jaccard_scores, detailed_analysis) where jaccard_scores is a list of similarity scores
               and detailed_analysis is a list of dictionaries with detailed comparison information
    """
    jaccard_scores = []
    detailed_analysis = []
    for idx, row in df.iterrows():
        # Collect all non-empty human annotations (codes, need, element)
        human_items = [row.get(f'code_{i}_human', 'None') for i in range(1, 6) if row.get(f'code_{i}_human', 'None') != 'None']
        if row.get('need', None) not in (None, 'None', 'null'):
            human_items.append(str(row.get('need')))
        if row.get('element', None) not in (None, 'None', 'null'):
            human_items.append(str(row.get('element')))

        # Collect all non-empty model annotations (codes, need, element)
        model_items = [row.get(f'code_{i}_model', 'None') for i in range(1, 6) if row.get(f'code_{i}_model', 'None') != 'None']
        if row.get('need_model', None) not in (None, 'None', 'null'):
            model_items.append(str(row.get('need_model')))
        if row.get('element_model', None) not in (None, 'None', 'null'):
            model_items.append(str(row.get('element_model')))

        # Get video ID for reference
        hashed_id = row.get('hashed_videoId', None)
        
        # Calculate Jaccard similarity between human and model annotations
        similarity = jaccard_similarity(human_items, model_items)
        jaccard_scores.append(similarity)
        
        # Store detailed comparison information for this row
        detailed_analysis.append({
            'hashed_videoId': hashed_id,
            'category': row.get('category', None),
            'human_items': human_items,
            'model_items': model_items,
            'intersection': set(human_items).intersection(set(model_items)),
            'union': set(human_items).union(set(model_items)),
            'jaccard_score': similarity
        })
    
    # Print summary statistics of Jaccard similarity scores
    print(f"Average Jaccard Similarity: {np.mean(jaccard_scores):.4f}")
    print(f"Median Jaccard Similarity: {np.median(jaccard_scores):.4f}")
    print(f"Standard Deviation: {np.std(jaccard_scores):.4f}")
    print(f"Min Jaccard Score: {np.min(jaccard_scores):.4f}")
    print(f"Max Jaccard Score: {np.max(jaccard_scores):.4f}")
    
    return jaccard_scores, detailed_analysis

jaccard_scores, detailed_analysis = calculate_jaccard_analysis(combined_human_model_codes)

# Data Analysis

## Step1. Load joined_needs_elements_codes.csv file for the analysis

In [ ]:
import pandas as pd
import ast

# Read the joined dataframe
joined_path = '../csv/joined_needs_elements_codes.csv'
df = pd.read_csv(joined_path)

# Ensure 'codes' column is a list (if not, parse it)
def parse_codes(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return val

df['codes'] = df['codes'].apply(parse_codes)

# Ensure 'Quarter' is present (should be, else create)
if 'Quarter' not in df.columns and 'dateTime' in df.columns:
    df['date'] = pd.to_datetime(df['dateTime'], errors='coerce')
    df['Quarter'] = df['date'].dt.to_period('Q')


## Step 2: Code Co-occurrence: Jaccard Calculation

The following function calculates co-occurrence matrices using the Jaccard index by element, need, and quarter.


In [ ]:
from itertools import combinations
import numpy as np

def build_jaccard_scores_over_time(df):
    all_scores = []
    for (element, quarter), group in df.groupby(['element', 'Quarter']):
        code_sets = [set(c) for c in group['codes'] if isinstance(c, (list, set))]
        all_codes = set.union(*code_sets) if code_sets else set()
        
        # Count code occurrences
        code_counts = {code: sum([code in codes for codes in code_sets]) for code in all_codes}
        cooccurrences = {}
        
        for codes in code_sets:
            for code1, code2 in combinations(sorted(list(codes)), 2):
                pair = tuple(sorted((code1, code2)))
                cooccurrences[pair] = cooccurrences.get(pair, 0) + 1
        
        for (code1, code2), intersection in cooccurrences.items():
            union = code_counts.get(code1, 0) + code_counts.get(code2, 0) - intersection
            jaccard = intersection / union if union else 0.0
            all_scores.append({
                'element': element,
                'quarter': str(quarter),
                'code_pair': f"{code1} & {code2}",
                'jaccard': jaccard
            })
    return all_scores


In [ ]:

def export_jaccard_to_csv(jaccard_scores, filename_prefix="jaccard_table"):
    df = pd.DataFrame(jaccard_scores)
    if df.empty:
        print("No data to export.")
        return
    
    for element in df['element'].unique():
        element_df = df[df['element'] == element]
        pivot = element_df.pivot(index='code_pair', columns='quarter', values='jaccard')
        pivot = pivot.reindex(sorted(pivot.columns), axis=1)
        csv_filename = f"{filename_prefix}_{element}.csv"
        pivot.to_csv(csv_filename)
        print(f"Saved {csv_filename}")

# Example usage:
scores = build_jaccard_scores_over_time(df)
export_jaccard_to_csv(scores)


## Step 3: Visualisation of Temporal Trends

The following cells plot rolling-average Jaccard trends per element/need, revealing content shifts and memetic collectivisation.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_smoothed_jaccard_heatmap(jaccard_scores,
                                  target_element,
                                  rolling_window=2,
                                  min_max_jaccard=0.06,
                                  top_n_pairs=10,
                                  start_quarter='2020-Q1'):
    """
    Plot a clean temporal heatmap:
    - filter to one element
    - smooth over time
    - keep only strongest code pairs
    - restrict to quarters from start_quarter onwards
    """
    scores_df = pd.DataFrame(jaccard_scores)

    # 1) Filter for the requested element
    element_df = scores_df[scores_df['element'] == target_element].copy()
    if element_df.empty:
        print(f"No data available to plot for element: {target_element}")
        return

    # 2) Filter by quarter: only keep quarters >= start_quarter
    # Assumes 'quarter' column is string like '2020-Q1', '2020-Q2', etc.
    element_df = element_df[element_df['quarter'] >= start_quarter]

    # 3) Pivot: rows = code_pair, columns = quarter
    heatmap_base = element_df.pivot_table(
        index='code_pair',
        columns='quarter',
        values='jaccard'
    ).fillna(0)

    # Ensure quarters sorted chronologically
    heatmap_base = heatmap_base.reindex(sorted(heatmap_base.columns), axis=1)

    # 4) Smooth across time (rolling over columns, using transpose)
    smoothed = (
        heatmap_base.T
        .rolling(window=rolling_window, min_periods=1)
        .mean()
        .T
    )

    # 5) Compute the strength of each pair (max Jaccard over all quarters)
    pair_strength = smoothed.max(axis=1)

    # Keep only pairs that ever exceed min_max_jaccard
    strong_pairs = pair_strength[pair_strength >= min_max_jaccard].sort_values(ascending=False)

    if strong_pairs.empty:
        print(f"No code pairs exceed min_max_jaccard={min_max_jaccard} for element {target_element}.")
        return

    # Optionally limit to top N strongest pairs
    if top_n_pairs is not None:
        strong_pairs = strong_pairs.head(top_n_pairs)

    smoothed_heatmap = smoothed.loc[strong_pairs.index]

    # 6) Plot
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(18, 10))

    sns.heatmap(
        smoothed_heatmap,
        ax=ax,
        cmap='YlOrRd',
        linewidths=.5,
        annot=True,
        fmt=".2f",
        cbar_kws={'label': f'Jaccard Score ({rolling_window}-Quarter Rolling Avg)'}
    )

    ax.set_title(f"Smoothed Temporal Trend of Jaccard Scores for: {target_element}", fontsize=16)
    ax.set_xlabel("Time (Quarter)", fontsize=12)
    ax.set_ylabel("Code Pair", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## Display Scores per Element

In [ ]:
plot_smoothed_jaccard_heatmap(
    jaccard_scores,
    target_element='Worker',
    rolling_window=2,
    min_max_jaccard=0.06,
    top_n_pairs=10,
    start_quarter='2020-Q1'
)


## Step 4. Perform Permutation Test

In [ ]:

def fast_permutation_test(df, element='Migrant', n_permutations=1000, target_pairs=None, random_state=None):
    # Filter data
    sub_df = df[df['element'] == element].copy()
    codes_lists = sub_df['codes'].tolist()
    code_set = sorted({code for codes in codes_lists for code in codes})
    
    # Get all code pairs (or only those user-specified)
    if target_pairs is None:
        pairs = list(combinations(code_set, 2))
    else:
        pairs = target_pairs
        
    # Prepare, for each doc, the set of unique codes
    code_presence = [{code for code in codes} for codes in codes_lists]
    n = len(code_presence)

    # Observed co-occur and singleton counts per pair
    pair_obs = {}
    for c1, c2 in pairs:
        both = sum((c1 in doc) and (c2 in doc) for doc in code_presence)
        either = sum(((c1 in doc) or (c2 in doc)) for doc in code_presence)
        pair_obs[(c1, c2)] = both / either if either > 0 else 0

    # Build a flat list of all codes, and a matching list of code counts per doc
    all_codes = [c for codes in codes_lists for c in codes]
    counts_per_doc = [len(codes) for codes in codes_lists]

    observed_matrix = pd.DataFrame(0, index=code_set, columns=code_set, dtype=float)
    for (c1, c2), val in pair_obs.items():
        observed_matrix.loc[c1, c2] = val
        observed_matrix.loc[c2, c1] = val
    
    # Permutation accumulation
    rng = np.random.default_rng(random_state)
    count_matrix = {pair: [] for pair in pairs}

    for _ in range(n_permutations):
        rng.shuffle(all_codes)
        idx = 0
        perm_codes_docs = []
        for k in counts_per_doc:
            perm_codes_docs.append(set(all_codes[idx:idx + k]))
            idx += k

        for (c1, c2) in pairs:
            both = sum((c1 in doc) and (c2 in doc) for doc in perm_codes_docs)
            either = sum(((c1 in doc) or (c2 in doc)) for doc in perm_codes_docs)
            val = both / either if either > 0 else 0
            count_matrix[(c1, c2)].append(val)
    
    # Calculate mean and empirical p-values
    perm_mean_vals = {pair: np.mean(vals) for pair, vals in count_matrix.items()}
    p_values = {pair: np.mean([v >= pair_obs[pair] for v in count_matrix[pair]]) for pair in pairs}
    
    perm_mean_df = pd.DataFrame(0, index=code_set, columns=code_set, dtype=float)
    p_vals_df = pd.DataFrame(1, index=code_set, columns=code_set, dtype=float)
    for (c1, c2) in pairs:
        perm_mean_df.loc[c1, c2] = perm_mean_vals[(c1, c2)]
        perm_mean_df.loc[c2, c1] = perm_mean_vals[(c1, c2)]
        p_vals_df.loc[c1, c2] = p_values[(c1, c2)]
        p_vals_df.loc[c2, c1] = p_values[(c1, c2)]

    return observed_matrix, perm_mean_df, p_vals_df

# USAGE EXAMPLE
# For only key pairs (recommend for speed)
target_pairs = [('DR-PSC', 'TA-DI'), ('DR-PSC', 'TA-RL'), ('ET-IC', 'DR-PSC'), ('ET-IC', 'TA-DI')] #Target pairs taken for the analysis
observed, perm_mean, p_vals = fast_permutation_test(df, element='Migrant', n_permutations=500, target_pairs=target_pairs)

# For all pairs (if speed is tolerable)
# observed, perm_mean, p_vals = fast_permutation_test(df, element='Worker', n_permutations=500)


## Step 5. Print the results of the permutation test

In [ ]:
# Example: For a given element and selected code pairs
element = 'Tourist'  # You can iterate over elements as needed
target_pairs = [ 
     ('DR-IE', 'DR-PSC'),
    ('DR-IE', 'DR-VS'),
    ('DR-IE', 'ET-PF'),
    ('DR-IE', 'TA-DI'),
    ('DR-PSC', 'DR-VS'),
    ('DR-PSC', 'ET-PF'),
    ('DR-PSC', 'TA-DI'),
    ('DR-PSC', 'TA-RL'),
    ('DR-VS', 'TA-DI'),
    ('ET-PF', 'TA-DI'),
    ('TA-DI', 'TA-RL')]  # Replace with your set
n_permutations = 1000

# Run permutation test (reuse your function)
observed, perm_mean, pvals = fast_permutation_test(df, element, n_permutations=n_permutations, target_pairs=target_pairs)

# Collate results and export to CSV
rows = []
for c1, c2 in target_pairs:
    rows.append({
        'element': element,
        'code_pair': f"{c1}-{c2}",
        'observed': observed.loc[c1, c2],
        'perm_mean': perm_mean.loc[c1, c2],
        'empirical_p': pvals.loc[c1, c2]
    })

results_df = pd.DataFrame(rows)
results_df.to_csv(f'permutation_test_results_{element}.csv', index=False)
print(f"Saved permutation test results for {element} to CSV.")


In [ ]:

for pair in target_pairs:
    c1, c2 = pair
    obs = observed.loc[c1, c2]
    mean_perm = perm_mean.loc[c1, c2]
    pval = p_vals.loc[c1, c2]
    print(f"{c1} & {c2} — Observed: {obs:.3f}, Perm mean: {mean_perm:.3f}, p-value: {pval:.3g}")


## Step 6. Pairwise Jaccard calculation per quarter and need

In [ ]:
from collections import defaultdict

# Assume your DataFrame is named df
# Create a proper copy of the filtered DataFrame to avoid SettingWithCopyWarning
df_worker = df[df['need'] == 'Basic'].copy()

# Make sure codes column contains tuples (hashable) instead of lists
# Convert lists to tuples if necessary
if df_worker['codes'].apply(lambda x: isinstance(x, list)).any():
    df_worker.loc[:, 'codes'] = df_worker['codes'].apply(lambda x: tuple(x) if isinstance(x, list) else x)

# Explode codes column so each row is a single code per video
df_worker_exploded = df_worker.explode('codes')

# Collect sets of codes by video and quarter
# Convert back to set after groupby since we need set operations later
video_codes = df_worker.groupby(['hashed_videoId', 'Quarter'])['codes'].apply(lambda x: set(code for codes in x for code in codes)).reset_index()

# Generate all code pairs seen in the dataset
# Flatten the codes and convert to a list of individual codes
all_codes = sorted(set(code for code_set in video_codes['codes'] for code in code_set))
code_pairs = list(combinations(all_codes, 2))

# Calculate Jaccard index for each code pair per quarter
records = []
for quarter, group in video_codes.groupby('Quarter'):
    code_sets = group['codes'].tolist()
    for code_a, code_b in code_pairs:
        inter = sum((code_a in codes) and (code_b in codes) for codes in code_sets)
        union = sum(((code_a in codes) or (code_b in codes)) for codes in code_sets)
        jaccard = inter / union if union else 0
        records.append({
            'Quarter': quarter,
            'Code_Pair': f"{code_a} & {code_b}",
            'Jaccard_Score': jaccard
        })
df_jaccard = pd.DataFrame(records)

## Step 7. Maslow Needs Projections by Elements

In [ ]:

def build_jaccard_matrices_over_time(df):
    """
    Builds Jaccard co-occurrence matrices for codes, grouped by element, need, and quarter.

    Args:
        df (pd.DataFrame): DataFrame with columns 'hashed_videoId', 'codes', 'element', 'need', 'quarter'

    Returns:
        dict: Keys are (element, need, quarter) tuples, values are co-occurrence matrices.
    """
    cooccurrence_matrices = {}
    grouped = df.groupby(['element', 'need', 'Quarter'])

    for (element, need, quarter), group in grouped:
        all_codes = set()
        for codes in group['codes']:
            all_codes.update(codes)
        all_codes_list = sorted(all_codes)
        if len(all_codes_list) < 2:
            continue

        matrix = pd.DataFrame(0, index=all_codes_list, columns=all_codes_list, dtype=float)
        code_counts = {code: 0 for code in all_codes_list}

        for codes_list in group['codes']:
            unique_codes = set(codes_list)
            for code in unique_codes:
                code_counts[code] += 1
            for code1, code2 in combinations(unique_codes, 2):
                matrix.loc[code1, code2] += 1
                matrix.loc[code2, code1] += 1

        for i in all_codes_list:
            for j in all_codes_list:
                if i == j:
                    continue
                intersection = matrix.loc[i, j]
                union = code_counts[i] + code_counts[j] - intersection
                matrix.loc[i, j] = intersection / union if union > 0 else 0.0
        np.fill_diagonal(matrix.values, 0)
        cooccurrence_matrices[(element, need, quarter)] = matrix

    return cooccurrence_matrices

def plot_jaccard_trends_by_element(cooccurrence_matrices_over_time):
    trend_data = []
    for (element, need, quarter), matrix in cooccurrence_matrices_over_time.items():
        if matrix.empty or matrix.shape[0] < 2:
            avg_score = 0
        else:
            num_pairs = matrix.shape[0] * (matrix.shape[1] - 1)
            total_jaccard_sum = matrix.values.sum()
            avg_score = total_jaccard_sum / num_pairs if num_pairs > 0 else 0
        trend_data.append({'quarter': quarter, 'need': need, 'element': element, 'average_jaccard': avg_score})

    if not trend_data:
        print("No data available to plot.")
        return

    trends_df = pd.DataFrame(trend_data)
    trends_df = trends_df.sort_values('quarter')
    trends_df = trends_df[trends_df['quarter'] >= '2020Q1']

    elements = sorted(trends_df['element'].unique())
    desired_order = ['Basic', 'Safety', 'Esteem', 'Social', 'Self-actualization']
    num_elements = len(elements)
    ncols, nrows = 2, 2  # Adjust if needed
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 12), sharex=True, sharey=True)
    for i, element in enumerate(elements):
        ax = axes[i // ncols, i % ncols]
        group_df = trends_df[trends_df['element'] == element]
        pivot_df = group_df.pivot_table(
            index='quarter', columns='need', values='average_jaccard', aggfunc='mean'
        )
        pivot_df = pivot_df[[col for col in desired_order if col in pivot_df.columns]]
        rolling_avg_df = pivot_df.rolling(window=3, min_periods=1).mean()
        rolling_avg_df.index = pd.PeriodIndex(rolling_avg_df.index, freq='Q').strftime('%Y-Q%q')
        rolling_avg_df.plot(ax=ax, marker='o')
        ax.set_title(f'Average Jaccard Score Trends by Need for {element}', fontsize=14)
        ax.set_xlabel('Quarter')
        ax.set_ylabel('Avg. Jaccard Score')
        ax.legend(title='Need Category', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True)
    plt.tight_layout()
    plt.show()

# Use directly with your dataframe
jaccard_matrices_by_quarter = build_jaccard_matrices_over_time(df)
plot_jaccard_trends_by_element(jaccard_matrices_by_quarter)


## Step 8. Jaccard Scores for Element-Need-Quarter

In [ ]:

from itertools import combinations
import pandas as pd
import numpy as np

records = []

for (element, need, quarter), group in df.groupby(['element', 'need', 'Quarter']):
    # Gather all codes used by this group
    codesets = group['codes'].tolist()
    allcodes = set(code for codes in codesets for code in codes)
    if len(allcodes) < 2:
        avgscore = 0.0
    else:
        jaccards = []
        for codea, codeb in combinations(allcodes, 2):
            inter = sum((codea in codes and codeb in codes) for codes in codesets)
            union = sum((codea in codes or codeb in codes) for codes in codesets)
            score = inter / union if union else 0.0
            jaccards.append(score)
        avgscore = np.mean(jaccards) if jaccards else 0.0
    records.append({
        'element': element,
        'need': need,
        'Quarter': quarter,
        'AverageJaccardScore': avgscore
    })

avg_jaccard_trend_df = pd.DataFrame(records)
avg_jaccard_trend_df.to_csv("../csv/avg_jaccard_trend_by_need_per_element.csv", index=False)
print("Saved average Jaccard Score Trend by Need per Element to CSV.")


## Step 9. Mixed Effects Modeling

In [ ]:

import statsmodels.formula.api as smf

model = smf.mixedlm("Jaccard_Score ~ Quarter_num", df_agg, groups=df_agg['element_need'])
result = model.fit()
print(result.summary())


In [ ]:
#Mixed Effects As per a particular group

group = 'Migrant_Social'
# Use .iloc[0] to safely access the scalar value
random_intercept = result.random_effects.get(group, pd.Series([0])).iloc[0]

# Fixed effects as before
fixed_intercept = result.fe_params['Intercept']
fixed_slope = result.fe_params['Quarter_num']

intercept_for_group = fixed_intercept + random_intercept

print(f"For {group}, Intercept: {intercept_for_group}, Slope (Quarter_num): {fixed_slope}")


In [ ]:
# Initialize empty list to store results for each group
output = []
# Get the fixed effects parameters from the mixed model result
fixed_intercept = result.fe_params['Intercept']
fixed_slope = result.fe_params['Quarter_num']

# Each group has its own random intercept
for group in result.random_effects.keys():
    # Extract the random intercept effect for this group
    random_intercept = result.random_effects[group].iloc[0]  # Take first value, typically intercept
    # Calculate the total intercept for this group (fixed + random effect)
    intercept = fixed_intercept + random_intercept
    # Store the group's information in a dictionary
    output.append({
        'element_need': group,
        'group_intercept': intercept,
        'fixed_slope_Quarter_num': fixed_slope
    })

# Convert the list of dictionaries to a DataFrame
summary_df = pd.DataFrame(output)
# Save the results to a CSV file
summary_df.to_csv("../csv/mixed_model_group_coefficients.csv", index=False)
print("Saved mixed model group intercepts and slope (Quarter_num) to CSV.")

## Instructions & Reproducibility

- Requirements: Python 3.9.21, pandas, seaborn, matplotlib, statsmodels, itertools
- Data: Upload `joined_needs_elements_codes.csv` with the specified column structure.
- To reproduce: "Restart Kernel & Run All Cells".
- All code is self-contained.
